In [3]:
pip install pandas numpy matplotlib seaborn scikit-learn

  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl (510 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
  Using cached pillow-12.1.1-cp310-cp310-win_amd64.whl (7.0 MB)
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl (41.3 MB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\asmis\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [4]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import joblib
import json

In [10]:
df = pd.read_csv(r"C:\Users\asmis\OneDrive\Desktop\foodbridge\data\food_master_dataset.csv")

df.head()

,date,city,district,outlet_type,total_prepared_kg,total_sold_kg,surplus_kg,footfall_count,rainfall_mm,temperature_c,...,unemployment_rate,malnutrition_rate,literacy_rate,population_density,rainfall_category,temperature_category,surplus_category,feedback_source,feedback_text,sentiment_label
0,2024-04-12,Chennai,Mumbai Suburban,restaurant,234.27,178.91,55.36,936,33.819924,22.980879,...,9.281988,23.345097,71.159051,13857.50378,medium,cool,medium,logistics_team,Food supply was slightly different than expect...,neutral
1,2025-03-11,Chennai,Hyderabad,restaurant,216.43,169.38,47.05,611,57.927890,27.431975,...,5.543214,31.585074,57.331012,27642.76583,medium,moderate,medium,restaurant_staff,Coordination with volunteers worked great and ...,positive
2,2024-09-27,Chennai,Chennai,restaurant,222.05,135.79,86.26,1004,19.304584,25.671839,...,2.607615,37.821608,74.279447,22704.52683,low,moderate,medium,restaurant_staff,Food supply was slightly different than expect...,neutral
3,2024-04-16,Hyderabad,Chennai,restaurant,351.51,329.97,21.54,787,10.248639,27.042423,...,5.059368,15.320457,74.441768,19956.61007,low,moderate,low,logistics_team,Community support was excellent and food reach...,positive
4,2024-03-12,Bangalore,Hyderabad,supermarket,406.52,355.98,50.54,805,23.895929,24.664987,...,13.216180,8.737902,85.915223,14806.84788,medium,cool,medium,volunteer,Shelter operations were routine with no major ...,neutral


In [11]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek

df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek

In [12]:
df["expected_demand"] = df["number_of_people"] * df["meals_per_person"]

df["sales_efficiency"] = df["footfall_count"] / (df["total_prepared_kg"] + 1)

df["demand_ratio"] = df["expected_demand"] / (df["total_prepared_kg"] + 1)

df["supply_gap"] = df["total_prepared_kg"] - df["expected_demand"]

df["weather_index"] = df["rainfall_mm"] * df["temperature_c"]

df["estimated_sales"] = df["footfall_count"] * df["meals_per_person"] * 0.5

df["prep_sales_gap"] = df["total_prepared_kg"] - df["estimated_sales"]

df["prep_per_person"] = df["total_prepared_kg"] / (df["number_of_people"] + 1)

In [13]:
categorical_cols = [
    "city",
    "district",
    "outlet_type",
    "rainfall_category",
    "temperature_category"
]

df = pd.get_dummies(df, columns=categorical_cols)

In [14]:
drop_cols = [
    "date",
    "feedback_text",
    "feedback_source",
    "sentiment_label",
    "surplus_category"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [15]:
y = df["surplus_kg"]

X = df.drop(columns=["surplus_kg"])

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
model = RandomForestRegressor(
    n_estimators=800,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,n_estimators,800
,criterion,'squared_error'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
pred = model.predict(X_test)

In [19]:
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("MAE:", mae)
print("R2:", r2)

MAE: 1.6893207132009211
R2: 0.9928941450257092


In [20]:
joblib.dump(model,r"C:\Users\asmis\OneDrive\Desktop\foodbridge\saved_models\foodbridge_regressor.pkl")

['C:\\Users\\asmis\\OneDrive\\Desktop\\foodbridge\\saved_models\\foodbridge_regressor.pkl']

In [22]:
features = X.columns.tolist()

with open(r"C:\Users\asmis\OneDrive\Desktop\foodbridge\saved_models\model_features.json","w") as f:
    json.dump(features,f)

In [23]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, pred))

print("RMSE:", rmse)

RMSE: 4.2333857645117465
